# PHO Chronic Disease EDA

**CIND 820 Big Data Analytics Project, Milestone 2: Architecture and Data Audit**

Mamkon Mercy Oyeleke | Student Number: 501279489 | Toronto Metropolitan University

**Data source:** Public Health Ontario - Chronic Disease Incidence and Prevalence Snapshot (2014-2023)
(https://www.publichealthontario.ca/en/Data-and-Analysis/Chronic-Disease/Chronic-Disease-Incidence-Prevalence)

**Outputs:** `../outputs/figures/fig1` through `fig7`, and `../outputs/tables/eda_summary_stats.csv`

Run the setup cell below first. It clones this repository when running in Google Colab, so the relative `../data/raw/` and `../outputs/` paths used throughout this notebook resolve correctly.

## Setup

In [ ]:
import os, sys

# If running in Google Colab, clone the repo so relative data/output paths resolve correctly
if 'google.colab' in sys.modules:
    if not os.path.exists('Architecture_and_Data_Audit'):
        os.system('git clone https://github.com/mamkon/Architecture_and_Data_Audit.git')
    os.chdir('Architecture_and_Data_Audit/notebooks')

print("Working directory:", os.getcwd())

## Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')

# ── Output directory ──────────────────────────────────────────────────────────
os.makedirs('../outputs/figures', exist_ok=True)
os.makedirs('../outputs/tables', exist_ok=True)

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 150,
})

PALETTE = ['#185FA5', '#1D9E75', '#D85A30', '#BA7517', '#993556', '#639922', '#534AB7', '#5F5E5A']

## Section 1: DATA LOADING AND BASIC PROFILING

In [ ]:
print("=" * 70)
print("SECTION 1 – LOADING AND PROFILING")
print("=" * 70)

FILE = '../data/raw/PHO_Chronic_Disease_Inc_Prev_Snapshot_2014_2023.xlsx'
df_raw = pd.read_excel(FILE, skiprows=2)

# Rename long suppression column
df_raw.rename(columns={df_raw.columns[-1]: 'Suppression_Flag'}, inplace=True)

print(f"\nRaw dataset shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"\nColumn names:")
for col in df_raw.columns:
    print(f"  {col}")

print(f"\nYear range   : {df_raw['Year'].min()} – {df_raw['Year'].max()}")
print(f"Geographies  : {df_raw['Geography'].nunique()} unique PHUs")
print(f"Indicators   : {df_raw['Indicator'].nunique()} unique")
print(f"Measure types: {df_raw['Measure'].nunique()} unique")

## Section 2: MISSING VALUES AND SUPPRESSION AUDIT

In [ ]:
print("\n" + "=" * 70)
print("SECTION 2 – MISSING VALUES AND SUPPRESSION AUDIT")
print("=" * 70)

missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]
print("\nMissing value summary:")
print(missing_df.to_string())

suppressed = df_raw['Suppression_Flag'].notna().sum()
print(f"\nSuppressed records (§ flag) : {suppressed}")
print(f"Suppression rate            : {suppressed / len(df_raw) * 100:.3f}%")

# Suppression by indicator
supp_by_indicator = df_raw[df_raw['Suppression_Flag'].notna()]['Indicator'].value_counts()
print("\nSuppressed records by indicator:")
print(supp_by_indicator.to_string())

# ── Fig 2: Suppression analysis ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Data Suppression and Completeness Audit', fontsize=14, fontweight='bold')

# Left: missing value bar chart
short_cols = {
    'Rate': 'Rate',
    'Cases': 'Cases',
    'Population': 'Population',
    '95% Confidence Interval': '95% CI',
    'Significance Compared to Ontario': 'Significance',
}
miss_plot = missing_df.copy()
miss_plot.index = [short_cols.get(i, i) for i in miss_plot.index]
axes[0].barh(miss_plot.index, miss_plot['Missing %'], color='#D85A30', edgecolor='white')
axes[0].set_xlabel('Missing (%)')
axes[0].set_title('Missing Values by Column')
for i, v in enumerate(miss_plot['Missing %']):
    axes[0].text(v + 0.1, i, f'{v:.1f}%', va='center', fontsize=9)

# Right: suppression by indicator
if len(supp_by_indicator) > 0:
    short_ind = [i.replace('Incidence of ', 'Inc. ').replace('Prevalence of ', 'Prev. ')
                   .replace(' in adults 20+', '') for i in supp_by_indicator.index]
    axes[1].barh(short_ind, supp_by_indicator.values, color='#BA7517', edgecolor='white')
    axes[1].set_xlabel('Count')
    axes[1].set_title('Suppressed Records by Indicator')
else:
    axes[1].text(0.5, 0.5, 'No suppression flags\ndetected', ha='center', va='center',
                 transform=axes[1].transAxes, color='gray')
    axes[1].set_title('Suppressed Records by Indicator')

plt.tight_layout()
plt.savefig('../outputs/figures/fig2_suppression_analysis.png', bbox_inches='tight')
plt.show()
print("  Saved: fig2_suppression_analysis.png")

## Section 3: FILTER TO ANALYTICAL SUBSET

In [ ]:
print("\n" + "=" * 70)
print("SECTION 3 – ANALYTICAL SUBSET")
print("=" * 70)

# Focus on age-standardized rates (both sexes) – key measure for cross-regional comparison
df = df_raw[df_raw['Measure'] == 'Age-standardized rate (both sexes)'].copy()
df = df[df['Suppression_Flag'].isna()]  # exclude suppressed records
df = df.dropna(subset=['Rate'])

# Map indicators to short names
indicator_map = {
    'Incidence of asthma': 'Asthma Incidence',
    'Prevalence of asthma': 'Asthma Prevalence',
    'Incidence of COPD in adults 20+': 'COPD Incidence',
    'Prevalence of COPD in adults 20+': 'COPD Prevalence',
    'Incidence of diabetes in adults 20+': 'Diabetes Incidence',
    'Prevalence of diabetes in adults 20+': 'Diabetes Prevalence',
    'Incidence of hypertension in adults 20+': 'Hypertension Incidence',
    'Prevalence of hypertension in adults 20+': 'Hypertension Prevalence',
}
df['Indicator_Short'] = df['Indicator'].map(indicator_map)

print(f"\nAnalytical subset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Indicators retained     : {df['Indicator_Short'].nunique()}")
print(f"Geographies retained    : {df['Geography'].nunique()}")
print(f"Years retained          : {sorted(df['Year'].unique())}")

## Section 4: DESCRIPTIVE STATISTICS

In [ ]:
print("\n" + "=" * 70)
print("SECTION 4 – DESCRIPTIVE STATISTICS BY INDICATOR")
print("=" * 70)

summary = df.groupby('Indicator_Short')['Rate'].agg(
    ['count', 'mean', 'median', 'std', 'min', 'max']
).round(1)
summary.columns = ['N', 'Mean', 'Median', 'Std Dev', 'Min', 'Max']
print("\n", summary.to_string())

summary.to_csv('../outputs/tables/eda_summary_stats.csv')
print("\n  Saved: eda_summary_stats.csv")

# ── Fig 1: Dataset overview ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('PHO Chronic Disease Dataset – Overview', fontsize=14, fontweight='bold')

# Top-left: record count per indicator
ind_counts = df['Indicator_Short'].value_counts()
short_names = [n.replace(' Incidence', '\nIncidence').replace(' Prevalence', '\nPrevalence')
               for n in ind_counts.index]
axes[0, 0].barh(short_names, ind_counts.values, color=PALETTE[:len(ind_counts)], edgecolor='white')
axes[0, 0].set_xlabel('Records')
axes[0, 0].set_title('Records per Indicator')

# Top-right: records per year
year_counts = df.groupby('Year').size()
axes[0, 1].bar(year_counts.index, year_counts.values, color='#185FA5', edgecolor='white')
axes[0, 1].set_xlabel('Year')
axes[0, 1].set_ylabel('Records')
axes[0, 1].set_title('Records per Year')
axes[0, 1].set_xticks(year_counts.index)
axes[0, 1].tick_params(axis='x', rotation=45)

# Bottom-left: rate distribution by indicator (box plot)
prevalence_inds = [i for i in df['Indicator_Short'].unique() if 'Prevalence' in i]
df_prev = df[df['Indicator_Short'].isin(prevalence_inds)]
short_prev = {i: i.replace(' Prevalence', '') for i in prevalence_inds}
df_prev = df_prev.copy()
df_prev['Ind_Short'] = df_prev['Indicator_Short'].map(short_prev)
axes[1, 0].boxplot(
    [df_prev[df_prev['Ind_Short'] == i]['Rate'].values for i in df_prev['Ind_Short'].unique()],
    labels=df_prev['Ind_Short'].unique(),
    patch_artist=True,
    boxprops=dict(facecolor='#E6F1FB', color='#185FA5'),
    medianprops=dict(color='#D85A30', linewidth=2),
    whiskerprops=dict(color='#185FA5'),
    capprops=dict(color='#185FA5'),
)
axes[1, 0].set_ylabel('Age-standardized rate (per 100,000)')
axes[1, 0].set_title('Rate Distribution – Prevalence Indicators')
axes[1, 0].tick_params(axis='x', rotation=15)

# Bottom-right: significance distribution
sig_counts = df['Significance Compared to Ontario'].value_counts()
colors_sig = {'Higher': '#D85A30', 'Lower': '#1D9E75', 'No': '#888780'}
bar_colors = [colors_sig.get(s, '#888780') for s in sig_counts.index]
axes[1, 1].bar(sig_counts.index, sig_counts.values, color=bar_colors, edgecolor='white')
axes[1, 1].set_xlabel('Significance vs Ontario')
axes[1, 1].set_ylabel('Records')
axes[1, 1].set_title('Regional Significance vs Ontario Average')
patches = [mpatches.Patch(color=v, label=k) for k, v in colors_sig.items()]
axes[1, 1].legend(handles=patches, fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/figures/fig1_dataset_overview.png', bbox_inches='tight')
plt.show()
print("\n  Saved: fig1_dataset_overview.png")

## Section 5: PREVALENCE TRENDS OVER TIME (ONTARIO-LEVEL)

In [ ]:
print("\n" + "=" * 70)
print("SECTION 5 – PREVALENCE TRENDS 2014–2023")
print("=" * 70)

# Ontario-level = median across all PHUs per year
ontario_trends = df.groupby(['Indicator_Short', 'Year'])['Rate'].median().reset_index()
prevalence_only = ontario_trends[ontario_trends['Indicator_Short'].str.contains('Prevalence')]

fig, ax = plt.subplots(figsize=(12, 6))
for i, ind in enumerate(prevalence_only['Indicator_Short'].unique()):
    subset = prevalence_only[prevalence_only['Indicator_Short'] == ind]
    label = ind.replace(' Prevalence', '')
    ax.plot(subset['Year'], subset['Rate'], marker='o', linewidth=2,
            color=PALETTE[i], label=label, markersize=5)

ax.set_xlabel('Year')
ax.set_ylabel('Median age-standardized rate (per 100,000)')
ax.set_title('Chronic Disease Prevalence Trends – Ontario Median Across PHUs (2014–2023)')
ax.legend(title='Disease', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.set_xticks(range(2014, 2024))
ax.tick_params(axis='x', rotation=45)
ax.axvspan(2019.5, 2021.5, alpha=0.08, color='gray', label='COVID-19 period')
ax.text(2020.2, ax.get_ylim()[0] + 10, 'COVID-19\nperiod', fontsize=8, color='gray')
plt.tight_layout()
plt.savefig('../outputs/figures/fig3_prevalence_trends_ontario.png', bbox_inches='tight')
plt.show()
print("  Saved: fig3_prevalence_trends_ontario.png")

## Section 6: REGIONAL BURDEN HEATMAP (2023 SNAPSHOT)

In [ ]:
print("\n" + "=" * 70)
print("SECTION 6 – REGIONAL BURDEN HEATMAP (2023)")
print("=" * 70)

df_2023 = df[(df['Year'] == 2023) & (df['Indicator_Short'].str.contains('Prevalence'))].copy()
pivot = df_2023.pivot_table(index='Geography', columns='Indicator_Short', values='Rate')
pivot.columns = [c.replace(' Prevalence', '') for c in pivot.columns]

# Drop rows with too many missing values
pivot = pivot.dropna(thresh=2)

# Normalise for heatmap
pivot_norm = (pivot - pivot.min()) / (pivot.max() - pivot.min())

fig, ax = plt.subplots(figsize=(10, 14))
sns.heatmap(
    pivot_norm,
    ax=ax,
    cmap='RdYlGn_r',
    linewidths=0.3,
    linecolor='white',
    annot=False,
    cbar_kws={'label': 'Normalised burden score (0 = lowest, 1 = highest)'},
)
ax.set_title('Regional Chronic Disease Burden – Normalised Prevalence Rates (2023)', pad=12)
ax.set_xlabel('')
ax.set_ylabel('')
ax.tick_params(axis='x', rotation=30)
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.savefig('../outputs/figures/fig4_regional_burden_heatmap.png', bbox_inches='tight')
plt.show()
print("  Saved: fig4_regional_burden_heatmap.png")

## Section 7: TOP AND BOTTOM REGIONS BY COMPOSITE BURDEN

In [ ]:
print("\n" + "=" * 70)
print("SECTION 7 – TOP AND BOTTOM REGIONS BY COMPOSITE BURDEN (2023)")
print("=" * 70)

# Composite score = mean normalised rate across all prevalence indicators
composite = pivot_norm.mean(axis=1).sort_values(ascending=False)
top5 = composite.head(5)
bottom5 = composite.tail(5)

print("\nTop 5 highest burden regions (2023):")
print(top5.to_string())
print("\nBottom 5 lowest burden regions (2023):")
print(bottom5.to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
fig.suptitle('Regional Chronic Disease Burden – Top and Bottom 5 PHUs (2023)', fontsize=13, fontweight='bold')

axes[0].barh(top5.index[::-1], top5.values[::-1], color='#D85A30', edgecolor='white')
axes[0].set_title('Highest Burden Regions')
axes[0].set_xlabel('Composite burden score (normalised)')
axes[0].tick_params(axis='y', labelsize=9)

axes[1].barh(bottom5.index[::-1], bottom5.values[::-1], color='#1D9E75', edgecolor='white')
axes[1].set_title('Lowest Burden Regions')
axes[1].set_xlabel('Composite burden score (normalised)')
axes[1].tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.savefig('../outputs/figures/fig5_top_bottom_regions.png', bbox_inches='tight')
plt.show()
print("  Saved: fig5_top_bottom_regions.png")

## Section 8: CORRELATION BETWEEN DISEASE INDICATORS

In [ ]:
print("\n" + "=" * 70)
print("SECTION 8 – CORRELATION BETWEEN DISEASE INDICATORS")
print("=" * 70)

corr = pivot.corr(method='pearson').round(2)
print("\nPearson correlation matrix (2023 prevalence rates):")
print(corr.to_string())

fig, ax = plt.subplots(figsize=(8, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr,
    ax=ax,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    vmin=-1, vmax=1,
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Pearson r'},
)
ax.set_title('Correlation Between Chronic Disease Prevalence Rates by PHU (2023)')
plt.tight_layout()
plt.savefig('../outputs/figures/fig6_disease_correlation.png', bbox_inches='tight')
plt.show()
print("  Saved: fig6_disease_correlation.png")

## Section 9: SIGNIFICANCE ANALYSIS

In [ ]:
print("\n" + "=" * 70)
print("SECTION 9 – REGIONAL SIGNIFICANCE PATTERNS")
print("=" * 70)

# Proportion of indicators where each region is Higher vs Lower than Ontario
sig_pivot = df[df['Year'] == 2023].pivot_table(
    index='Geography',
    columns='Significance Compared to Ontario',
    values='Rate',
    aggfunc='count'
).fillna(0)

sig_pivot = sig_pivot[['Higher', 'No', 'Lower']] if all(
    c in sig_pivot.columns for c in ['Higher', 'No', 'Lower']
) else sig_pivot

print("\n2023 significance count by region (Higher / No / Lower vs Ontario):")
print(sig_pivot.to_string())

fig, ax = plt.subplots(figsize=(10, 12))
sig_pivot.plot(kind='barh', stacked=True, ax=ax,
               color=['#D85A30', '#888780', '#1D9E75'], edgecolor='white')
ax.set_title('Count of Indicators Significantly Higher / Lower Than Ontario (2023)')
ax.set_xlabel('Number of indicators')
ax.set_ylabel('')
ax.legend(title='vs Ontario', loc='lower right', fontsize=9)
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.savefig('../outputs/figures/fig7_significance_distribution.png', bbox_inches='tight')
plt.show()
print("  Saved: fig7_significance_distribution.png")

## Section 10: DATA STRUCTURE NOTES FOR MILESTONE 2 REPORT

In [ ]:
print("\n" + "=" * 70)
print("SECTION 10 – DATA STRUCTURE NOTES FOR MILESTONE 2 REPORT")
print("=" * 70)

print("""
Dataset structure:
  - Long format: one row per Indicator x Measure x Year x Geography combination
  - Key join variable for CIHI linkage: 'Geography' (PHU name)
  - Key analytical variable: 'Rate' (age-standardized, per 100,000)
  - Time dimension: Year (2014–2023, 10 time points)

Data preparation steps required before joining with CIHI:
  1. Filter Measure == 'Age-standardized rate (both sexes)'
  2. Exclude suppressed records (Suppression_Flag == '§')
  3. Exclude records with missing Rate
  4. Standardise Geography names to match CIHI regional labels
  5. Pivot to wide format: rows = Geography x Year, cols = each indicator
  6. Align time periods with CIHI dataset (semi-annual vs annual)

Known limitations and impact on claims:
  - 7 records suppressed (small PHU cell counts) → cannot make claims for
    those specific PHU-year combinations
  - Data reflects conditions 1-2 years prior to publication date
  - Geography is PHU-level only → no sub-PHU (census division) analysis possible
  - Indicators limited to asthma, COPD, diabetes, hypertension
    → other chronic conditions (heart disease, stroke) not captured
  - Rate measure is age-standardized → raw counts differ significantly
    from rates in small PHUs
""")

print("=" * 70)
print("EDA COMPLETE - all figures and tables saved to ../outputs/")
print("=" * 70)